In [88]:
import numpy as np
from scipy.optimize import differential_evolution
from scipy.special import expit

In [89]:
def rw_transfer_llf(para,cue,outcome,action):
    
    """
    Transfer learning RW model
    
    args:
        para: parameter
        
        cue: happy/angry face, +0.5 denotes happy face, -0.5 denotes angry face
        
        outcome: reward/punishement, 1 denotes reward outcome, 0 denotes punishement outcome
        
        action: appraoch/avoidance, 1 denotes appraoch action, 0 denotes avoid action
    
    
    """
    
    alpha,tau,bias0,bias1,ep = para
    
    
    trial_len = len(action)
    nll = 0
    q_u = .5
    
    for t in range(trial_len):
        
        cue_t = cue[t]
        a_t = action[t]
        r_t = outcome[t]
        
        q_t = q_u if cue_t>0 else 1-q_u
        
        ## decision: e-softmax
        
        p_app = expit(tau * (2*q_t - 1) + bias0 + bias1 * cue_t)
        p_app = (1-ep) * p_app + ep/2
        
        
        # log-likelihood
        if a_t == 1:
            p_choice = p_app
        else:
            p_choice = 1 - p_app
        p_choice = np.clip(p_choice, 1e-8, 1 - 1e-8)
        nll -= np.log(p_choice)
        
        
        ## align outcome to happy face appraoch action
        
        rev_outcome = r_t if a_t == 1 else 1 - r_t
        cue_outcome = rev_outcome if cue_t > 0 else 1 - rev_outcome
        
        ## update
        
        q_u  += alpha * (cue_outcome - q_u)
    
    return nll


def rw_seprate_llf(para,cue,outcome,action):
    """
    Separate cue learning RW model
    
    args:
        para: parameter
        
        cue: happy/angry face, +0.5 denotes happy face, -0.5 denotes angry face
        
        outcome: reward/punishement, 1 denotes reward outcome, 0 denotes punishement outcome
        
        action: appraoch/avoidance, 1 denotes appraoch action, 0 denotes avoid action
    
    
    """
    
    alpha,tau,bias0,bias1,ep = para
    
    
    trial_len = len(action)
    nll = 0
    q_h = .5  ## happy face
    q_a = .5  ## angry face
    
    for t in range(trial_len):
        
        cue_t = cue[t]
        a_t = action[t]
        r_t = outcome[t]
        
        q_t = q_h if cue_t>0 else q_a
        
        ## decision: e-softmax
        
        p_app = expit(tau * (2*q_t - 1) + bias0 + bias1 * cue_t)
        p_app = (1-ep) * p_app + ep/2
        
        # log-likelihood
        if a_t == 1:
            p_choice = p_app
        else:
            p_choice = 1 - p_app
        p_choice = np.clip(p_choice, 1e-8, 1 - 1e-8)
        nll -= np.log(p_choice)
        
        
        ## align outcome to happy face appraoch action
        
        rev_outcome = r_t if a_t == 1 else 1 - r_t
        
        ## update
        
        pe = rev_outcome - q_t
        if cue[t] > 0:
            q_h += alpha*pe
        else:
            q_a += alpha*pe
    
    return nll


def fit_model(data):
    
    """
    Fit transfer learning and separate cue learning model
    
    args:
    
        data: list{'cue','outcome','action'}
    
    return:
        result: list('t_para','s_para','t_nll','s_nll')
    
    """
    cue = data['cue']
    outcome = data['outcome']
    action = data['choice']
    
    bounds = [
    (1e-4, 0.9999),   # alpha
    (1e-4, 20.0),    # tau
    (-8.0, 8.0),     # bias0
    (-8.0, 8.0),     # bias1
    (1e-4, 0.999)    # ep
    ]
    
    ## fit transfer learning model
    
    t_result = differential_evolution(
            rw_transfer_llf,
            bounds=bounds,
            args=(cue, outcome, action),
            maxiter=1000,
            popsize=30,
            polish=True,
            disp=True
            )
    
    ## fit separate cue learning model
    
    s_result = differential_evolution(
            rw_seprate_llf,
            bounds=bounds,
            args=(cue, outcome, action),
            maxiter=1000,
            popsize=30,
            polish=True,
            disp=True
            )
    
    return {'t_para':t_result.x,
           's_para':s_result.x,
           't_nll':t_result.fun,
           's_nll':s_result.fun}
     
    

In [107]:
## fit models to excluded and included participants
import pandas as pd
ex_dat = pd.read_csv('excluded_stan_fit.csv')
in_dat = pd.read_csv('included_session1_stan_fit.csv')

In [108]:
## select first two blocks
ex_dat['block1_2'] = ex_dat['block1'] + ex_dat['block2']
in_dat['block1_2'] = in_dat['block1'] + in_dat['block2']

ex_dat = ex_dat.loc[ex_dat['block1_2']>0]
in_dat = in_dat.loc[in_dat['block1_2']>0]

In [110]:
ex_grouped = (
    ex_dat.groupby('sub_id')[['choice', 'cue', 'outcome']]
    .apply(lambda df: {
        'choice': df['choice'].tolist(),
        'cue': df['cue'].tolist(),
        'outcome': df['outcome'].tolist()
    })
    .to_dict()
)
in_grouped = (
    in_dat.groupby('sub_id')[['choice','cue','outcome']]
    .apply(lambda df:{
        'choice':df['choice'].tolist(),
        'cue':df['cue'].tolist(),
        'outcome':df['outcome'].tolist()
    }).to_dict()
)

In [112]:
save_ex_fit = {}
save_in_fit = {}

for i,v in ex_grouped.items():
    save_ex_fit[i] = fit_model(v)
for i,v in in_grouped.items():
    save_in_fit[i] = fit_model(v)

differential_evolution step 1: f(x)= 264.7223485274729
differential_evolution step 2: f(x)= 264.7223485274729
differential_evolution step 3: f(x)= 262.56783052541203
differential_evolution step 4: f(x)= 262.56783052541203
differential_evolution step 5: f(x)= 262.56783052541203
differential_evolution step 6: f(x)= 262.56783052541203
differential_evolution step 7: f(x)= 262.56783052541203
differential_evolution step 8: f(x)= 262.3574867434304
differential_evolution step 9: f(x)= 262.3574867434304
differential_evolution step 10: f(x)= 262.3574867434304
differential_evolution step 11: f(x)= 262.3574867434304
differential_evolution step 12: f(x)= 262.2652510426013
Polishing solution with 'L-BFGS-B'
differential_evolution step 1: f(x)= 266.203046376312
differential_evolution step 2: f(x)= 266.0712535386588
differential_evolution step 3: f(x)= 264.9632518649679
differential_evolution step 4: f(x)= 264.8291910381357
differential_evolution step 5: f(x)= 264.8291910381357
differential_evolution 

In [115]:
t_nll = pd.DataFrame(save_in_fit).loc['t_nll',:]
s_nll = pd.DataFrame(save_in_fit).loc['s_nll',:]
s_ep = []
t_ep = []
for i in range(13):
    s_ep.append(1-pd.DataFrame(save_in_fit).loc['s_para',i+1][-1])
    t_ep.append(1-pd.DataFrame(save_in_fit).loc['t_para',i+1][-1])
pd.DataFrame({'transfer learning':t_nll,
             "separate cue learning":s_nll,
                          'transfer learning ep':t_ep,
             'separate learning ep':s_ep})

,transfer learning,separate cue learning,transfer learning ep,separate learning ep
1,181.685312,222.631344,0.836460,0.686299
2,181.386898,222.888162,0.998149,0.753343
3,216.265353,219.003797,0.797266,0.728815
4,228.715677,213.830017,0.691576,0.737925
5,226.69576,234.909335,0.918128,0.790783
6,128.950824,240.627034,0.879811,0.596786
7,216.908253,233.079924,0.999900,0.999900
8,238.683349,260.12524,0.715545,0.760753
9,171.757818,207.327364,0.952706,0.979478
10,208.119168,127.149154,0.999900,0.880981


In [116]:
t_nll = pd.DataFrame(save_ex_fit).loc['t_nll',:]
s_nll = pd.DataFrame(save_ex_fit).loc['s_nll',:]
s_ep = []
t_ep = []
for i in range(9):
    s_ep.append(1-pd.DataFrame(save_ex_fit).loc['s_para',i+1][-1])
    t_ep.append(1-pd.DataFrame(save_ex_fit).loc['t_para',i+1][-1])
pd.DataFrame({'transfer learning':t_nll,
             "separate cue learning":s_nll,
             'transfer learning ep':t_ep,
             'separate learning ep':s_ep})

,transfer learning,separate cue learning,transfer learning ep,separate learning ep
1,261.643649,264.5887,0.608958,0.407933
2,246.901791,266.712606,0.634343,0.426292
3,287.069156,285.708582,0.306598,0.303452
4,196.185877,211.991883,0.449677,0.216066
5,101.104334,99.897701,0.923080,0.927817
6,292.723835,287.188266,0.200066,0.235506
7,283.68883,279.937377,0.999900,0.999900
8,250.890489,256.283398,0.448191,0.999900
9,286.608565,278.960993,0.277061,0.345838
